# Semantic Embedding Extraction — PE File Dataset (RDset)

This notebook generates **384-dimensional sentence embeddings** from static PE (Portable Executable)
file features using `sentence-transformers/all-MiniLM-L6-v2`.

**Identifiers are dropped first.** `Name` and `md5` are removed immediately after loading, before any
other step. They identify a file rather than describe it, and in RDset `Name` also encodes the label:
every ransomware row is named `VirusShare_<hash>` and no benign row is, so embedding it lets the
model classify by file name alone. This matches the IEEE Access preprocessing (Kabuye et al., 2025).

**Approach:** Selected PE metadata columns (imports, exports, section entropy, resources, etc.) are
concatenated into a single string per sample. The string is then encoded by the transformer,
producing a dense vector used by the downstream Semantic Agent for ransomware classification.
All settings live in `data_pipeline/rdset_semantic.py`; this notebook runs that module step by step.

**Input:** `$SABRE_DATA_DIR/RDset/Ransomware.csv` (pipe-delimited, PE static features)  
**Output:** `$SABRE_DATA_DIR/RDset/semantic_dataset_with_embeddings.parquet`

## 1. Imports and Configuration

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "data_pipeline" else Path.cwd()))
from data_pipeline.rdset_semantic import (
    BATCH_SIZE, EMBEDDING_DIM, EMBEDDING_MODEL, IDENTIFIER_COLUMNS, SEMANTIC_COLUMNS,
    load_encoder, load_rdset, semantic_text,
)

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR       = Path(os.environ.get("SABRE_DATA_DIR", "/home/henry_k/Datasets"))
INPUT_CSV      = DATA_DIR / "RDset" / "Ransomware.csv"
OUTPUT_PARQUET = DATA_DIR / "RDset" / "semantic_dataset_with_embeddings.parquet"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device         : {device}")
print(f"Input          : {INPUT_CSV}")
print(f"Output         : {OUTPUT_PARQUET}")
print(f"Model          : {EMBEDDING_MODEL}")
print(f"Dropped first  : {IDENTIFIER_COLUMNS}")

Device         : cuda
Input          : /home/res-w23052945/TPM/datasets/RDset/Ransomware.csv
Output         : /home/res-w23052945/TPM/datasets/RDset/semantic_dataset_with_embeddings.parquet
Model          : sentence-transformers/all-MiniLM-L6-v2
Dropped first  : ['Name', 'md5']


## 2. Load and Prepare the Dataset

In [2]:
# load_rdset drops Name and md5 before anything else, adds `label`
# (1 = ransomware, from legitimate == 0) and drops the low-signal header fields.
df = load_rdset(INPUT_CSV)
assert not set(IDENTIFIER_COLUMNS) & set(df.columns)
print(f"Loaded         : {df.shape[0]:,} rows, {df.shape[1]} columns (identifiers removed)")
print(f"Columns        : {df.columns.tolist()}")

Loaded         : 138,047 rows, 40 columns (identifiers removed)
Columns        : ['Machine', 'Characteristics', 'SizeOfCode', 'SizeOfInitializedData', 'SizeOfUninitializedData', 'AddressOfEntryPoint', 'MajorOperatingSystemVersion', 'MinorOperatingSystemVersion', 'MajorImageVersion', 'MinorImageVersion', 'MajorSubsystemVersion', 'MinorSubsystemVersion', 'SizeOfImage', 'SizeOfHeaders', 'Subsystem', 'DllCharacteristics', 'SectionsNb', 'SectionsMeanEntropy', 'SectionsMinEntropy', 'SectionsMaxEntropy', 'SectionsMeanRawsize', 'SectionsMinRawsize', 'SectionMaxRawsize', 'SectionsMeanVirtualsize', 'SectionsMinVirtualsize', 'SectionMaxVirtualsize', 'ImportsNbDLL', 'ImportsNb', 'ImportsNbOrdinal', 'ExportNb', 'ResourcesNb', 'ResourcesMeanEntropy', 'ResourcesMinEntropy', 'ResourcesMaxEntropy', 'ResourcesMeanSize', 'ResourcesMinSize', 'ResourcesMaxSize', 'LoadConfigurationSize', 'VersionInformationSize', 'label']


In [3]:
print("Label distribution:")
print(df["label"].value_counts())
print("  0 = benign, 1 = ransomware")
df.head(3)

Label distribution:
label
1    96724
0    41323
Name: count, dtype: int64
  0 = benign, 1 = ransomware


,Machine,Characteristics,SizeOfCode,SizeOfInitializedData,SizeOfUninitializedData,AddressOfEntryPoint,MajorOperatingSystemVersion,MinorOperatingSystemVersion,MajorImageVersion,MinorImageVersion,...,ResourcesNb,ResourcesMeanEntropy,ResourcesMinEntropy,ResourcesMaxEntropy,ResourcesMeanSize,ResourcesMinSize,ResourcesMaxSize,LoadConfigurationSize,VersionInformationSize,label
0,332,258,361984,115712,0,6135,0,0,0,0,...,4,3.262823,2.568844,3.537939,8797.000000,216,18032,0,16,0
1,332,3330,130560,19968,0,81778,5,1,0,0,...,2,4.250461,3.420744,5.080177,837.000000,518,1156,72,18,0
2,332,3330,517120,621568,0,350896,5,1,0,0,...,11,4.426324,2.846449,5.271813,31102.272727,104,270376,72,18,0


## 3. Build the Semantic Text Field

The 19 PE metadata columns in `SEMANTIC_COLUMNS` are cast to strings and joined with a space.
This gives the transformer a compact, structured sentence per file
(e.g. `"332 8448 12 57 0 6.523 7.112 5.901 …"`). No identifier is part of the text.

Only columns that are actually present in the loaded CSV are used
(handles datasets where some fields may be missing).

In [4]:
present_semantic_cols = [c for c in SEMANTIC_COLUMNS if c in df.columns]
missing = set(SEMANTIC_COLUMNS) - set(present_semantic_cols)
if missing:
    print(f"[INFO] {len(missing)} semantic columns not found in CSV (will be skipped): {missing}")

df["semantic_text"] = semantic_text(df)

print(f"Using {len(present_semantic_cols)} semantic columns.")
print(f"\nSample semantic_text entries:")
for i, txt in enumerate(df["semantic_text"].iloc[:3]):
    print(f"  [{i}] {txt[:120]}")

Using 19 semantic columns.

Sample semantic_text entries:
  [0] 332 258 0 0 0 5.7668065537 7.22105072892 3.60742957555 4 3.26282271103 2.56884382364 3.53793936419 8797.0 216 18032 0 16
  [1] 332 3330 7 181 0 4.83968793753 6.56690933416 2.37352509596 2 4.2504605579 3.42074425303 5.08017686277 837.0 518 1156 72 
  [2] 332 3330 14 235 1 6.40955752803 7.60095678141 4.88519106848 11 4.42632398773 2.84644858861 5.27181275925 31102.2727273 1


## 4. Load the Sentence Transformer Model

In [5]:
print(f"Loading model: {EMBEDDING_MODEL}")
model = load_encoder(str(device))

print(f"Running on   : {device}")
print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")

Loading model: sentence-transformers/all-MiniLM-L6-v2


Running on   : cuda
Embedding dim: 384


/tmp/ipykernel_126753/404592089.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")


## 5. Generate Embeddings

In [6]:
texts = df["semantic_text"].tolist()
print(f"Encoding {len(texts):,} samples (batch_size={BATCH_SIZE})...")

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype(np.float32)  # [N, 384]

print(f"\nEmbedding matrix shape : {embeddings.shape}")
print(f"dtype                  : {embeddings.dtype}")
print(f"Mean L2 norm           : {np.linalg.norm(embeddings, axis=1).mean():.4f}")

Encoding 138,047 samples (batch_size=256)...


Batches:   0%|          | 0/540 [00:00<?, ?it/s]


Embedding matrix shape : (138047, 384)
dtype                  : float32
Mean L2 norm           : 1.0000


## 6. Attach Embeddings to the DataFrame

In [7]:
df["embedding"] = embeddings.tolist()

print(f"Final DataFrame shape  : {df.shape}")
print(f"Embedding vector length: {len(df['embedding'].iloc[0])}")
df[["label", "semantic_text", "embedding"]].head(2)

Final DataFrame shape  : (138047, 42)
Embedding vector length: 384


,label,semantic_text,embedding
0,0,332 258 0 0 0 5.7668065537 7.22105072892 3.607...,"[-0.010794051922857761, -0.0857551246881485, -..."
1,0,332 3330 7 181 0 4.83968793753 6.56690933416 2...,"[0.022753050550818443, -0.07142505794763565, -..."


## 7. Sanity Check — Embedding Separation

A quick cosine similarity check to confirm benign and ransomware embeddings
are separable in the embedding space.

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

benign_emb = embeddings[df["label"].values == 0][:100]
ransom_emb = embeddings[df["label"].values == 1][:100]

if len(benign_emb) >= 2 and len(ransom_emb) >= 2:
    sim_within_benign = cosine_similarity(benign_emb).mean()
    sim_within_ransom = cosine_similarity(ransom_emb).mean()
    sim_cross         = cosine_similarity(benign_emb, ransom_emb).mean()

    print(f"Within-benign similarity    : {sim_within_benign:.4f}")
    print(f"Within-ransomware similarity: {sim_within_ransom:.4f}")
    print(f"Cross-class similarity      : {sim_cross:.4f}")
    print("\n(Lower cross-class = better separation)")
else:
    print("[INFO] Not enough samples for similarity check.")

Within-benign similarity    : 0.9201
Within-ransomware similarity: 0.9259
Cross-class similarity      : 0.9187

(Lower cross-class = better separation)


## 8. Save to Parquet

In [9]:
Path(OUTPUT_PARQUET).parent.mkdir(parents=True, exist_ok=True)

df.to_parquet(OUTPUT_PARQUET, index=False)
print(f"Saved {len(df):,} rows → {OUTPUT_PARQUET}")

# Reload and verify
verify = pd.read_parquet(OUTPUT_PARQUET)
assert verify.shape == df.shape, f"[WARN] Shape mismatch: {verify.shape} != {df.shape}"
assert len(verify["embedding"].iloc[0]) == EMBEDDING_DIM, "[WARN] Unexpected embedding dimension!"
print(f"Reload check   : {verify.shape} — OK")
print(f"Columns saved  : {verify.columns.tolist()}")

Saved 138,047 rows → /home/res-w23052945/TPM/datasets/RDset/semantic_dataset_with_embeddings.parquet


Reload check   : (138047, 42) — OK
Columns saved  : ['Machine', 'Characteristics', 'SizeOfCode', 'SizeOfInitializedData', 'SizeOfUninitializedData', 'AddressOfEntryPoint', 'MajorOperatingSystemVersion', 'MinorOperatingSystemVersion', 'MajorImageVersion', 'MinorImageVersion', 'MajorSubsystemVersion', 'MinorSubsystemVersion', 'SizeOfImage', 'SizeOfHeaders', 'Subsystem', 'DllCharacteristics', 'SectionsNb', 'SectionsMeanEntropy', 'SectionsMinEntropy', 'SectionsMaxEntropy', 'SectionsMeanRawsize', 'SectionsMinRawsize', 'SectionMaxRawsize', 'SectionsMeanVirtualsize', 'SectionsMinVirtualsize', 'SectionMaxVirtualsize', 'ImportsNbDLL', 'ImportsNb', 'ImportsNbOrdinal', 'ExportNb', 'ResourcesNb', 'ResourcesMeanEntropy', 'ResourcesMinEntropy', 'ResourcesMaxEntropy', 'ResourcesMeanSize', 'ResourcesMinSize', 'ResourcesMaxSize', 'LoadConfigurationSize', 'VersionInformationSize', 'label', 'semantic_text', 'embedding']
